# Task 1: LoRA Fine-Tuning — Plain-English → JSON
**tndrr-ai-assignment · Nitin Singh · B.Tech CSE**

Fine-tunes `HuggingFaceTB/SmolLM2-360M-Instruct` with LoRA on a synthetic dataset  
that maps plain-English procurement requests to structured JSON `{action, item, quantity, date}`.

**⚠️ Before running:** Runtime → Change runtime type → **T4 GPU**  
**Estimated time:** ~15–20 min on T4

---
*Run all cells top-to-bottom.*

## Cell 1 — Install dependencies

In [ ]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes sentencepiece
print("✓ All packages installed.")

## Cell 2 — Generate the dataset

Creates **1,100 unique** plain-English → JSON examples.
- 1,000 → `data/train.jsonl`
- 100 → `data/test.jsonl`
- Fixed `SEED=42` for reproducibility. No train/test overlap.

In [ ]:
import json, os, random
from datetime import date, timedelta

SEED = 42
random.seed(SEED)

ACTIONS = ["order","buy","purchase","schedule","book","reserve","request","cancel","reorder","get"]
ITEMS   = ["laptop","monitor","keyboard","mouse","headset","webcam","desk","chair","notebook","pen",
           "charger","cable","tablet","phone","speaker","projector","whiteboard","coffee machine","water bottle","stapler"]
TEMPLATES = [
    "Please {action} {qty} {item}s by {date}.",
    "I need to {action} {qty} {item}(s) for {date}.",
    "Can you {action} {qty} {item}s? Deadline is {date}.",
    "We want to {action} {qty} units of {item} before {date}.",
    "Kindly {action} {qty} {item}s, delivery by {date}.",
    "Hey, {action} {qty} {item}s for me by {date}.",
    "I'd like to {action} {qty} {item}s, expected by {date}.",
    "Please go ahead and {action} {qty} {item}(s) — needed on {date}.",
    "Team needs {qty} {item}s — please {action} them by {date}.",
    "Urgent: {action} {qty} {item}s, latest by {date}.",
]

def _random_date():
    start = date(2025, 1, 1)
    return start + timedelta(days=random.randint(0, 364))

def _fmt(d): return d.strftime("%B %d, %Y")

def generate_all(n=1100):
    seen, examples = set(), []
    while len(examples) < n:
        tpl    = random.choice(TEMPLATES)
        action = random.choice(ACTIONS)
        item   = random.choice(ITEMS)
        qty    = random.randint(1, 50)
        d      = _random_date()
        date_s = _fmt(d)
        inp = tpl.format(action=action, qty=qty, item=item, date=date_s)
        if inp not in seen:
            seen.add(inp)
            examples.append({
                "input":  inp,
                "output": json.dumps({"action": action, "item": item,
                                       "quantity": qty, "date": date_s}),
            })
    return examples

os.makedirs("data", exist_ok=True)
all_ex = generate_all(1100)
random.seed(SEED); random.shuffle(all_ex)
train_ex, test_ex = all_ex[:1000], all_ex[1000:]

# Verify zero overlap
assert not ({e["input"] for e in train_ex} & {e["input"] for e in test_ex}), "OVERLAP DETECTED!"

for path, exs in [("data/train.jsonl", train_ex), ("data/test.jsonl", test_ex)]:
    with open(path, "w") as fh:
        for e in exs: fh.write(json.dumps(e) + "\n")

print(f"✓ train: {len(train_ex)} examples → data/train.jsonl")
print(f"✓ test : {len(test_ex)}  examples → data/test.jsonl")
print("Sample train[0]:", train_ex[0])
print("Sample test[0] :", test_ex[0])

## Cell 3 — LoRA Fine-Tuning
# ⚠️ RUN THIS — Requires T4 GPU (~15 min)

**LoRA hyperparameter rationale:**

| Param | Value | Reason |
|---|---|---|
| `r` | 8 | Enough capacity for 4-field JSON; small footprint |
| `lora_alpha` | 16 | 2×r — standard scaling heuristic from LoRA paper |
| `target_modules` | q_proj, v_proj | Attention projections capture instruction format |
| `lora_dropout` | 0.05 | Light regularisation for 1k-sample dataset |
| `lr` | 2e-4 | Standard LoRA learning rate; stable convergence |
| `epochs` | 3 | More risks memorisation on small dataset |
| `batch_size` | 4 × grad_accum 4 = 16 eff. | Fits T4 16 GB VRAM |

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# ⚠️ RUN THIS — LoRA Fine-Tuning on T4 GPU (~15 min)
# ═══════════════════════════════════════════════════════════════════
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    DataCollatorForSeq2Seq, Trainer, TrainingArguments,
)

MODEL_NAME   = "HuggingFaceTB/SmolLM2-360M-Instruct"
OUTPUT_DIR   = "./lora_model"
LORA_R       = 8
LORA_ALPHA   = 16
LORA_DROPOUT = 0.05
TARGET_MODS  = ["q_proj", "v_proj"]
LR           = 2e-4
EPOCHS       = 3
BATCH        = 4
GRAD_ACCUM   = 4
MAX_SEQ      = 256
SEED         = 42

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# Load tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True,
)
model.enable_input_require_grads()
model.gradient_checkpointing_enable()

# Apply LoRA
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODS,
    lora_dropout=LORA_DROPOUT, bias="none",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# Load data
import json
def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

train_data = load_jsonl("data/train.jsonl")
test_data  = load_jsonl("data/test.jsonl")
train_ds = Dataset.from_list(train_data)
test_ds  = Dataset.from_list(test_data)

def build_prompt(inp, out):
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        msgs = [{"role": "user", "content": inp}, {"role": "assistant", "content": out}]
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return f"Input: {inp}\nOutput: {out}"

def tokenize_fn(batch):
    texts = [build_prompt(i, o) for i, o in zip(batch["input"], batch["output"])]
    enc = tokenizer(texts, truncation=True, max_length=MAX_SEQ, padding=False)
    enc["labels"] = [ids[:] for ids in enc["input_ids"]]
    return enc

tok_train = train_ds.map(tokenize_fn, batched=True, remove_columns=["input","output"])
tok_test  = test_ds.map(tokenize_fn, batched=True, remove_columns=["input","output"])

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    seed=SEED,
    report_to="none",
    dataloader_num_workers=0,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True, pad_to_multiple_of=8)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=tok_train, eval_dataset=tok_test,
    data_collator=data_collator, tokenizer=tokenizer,
)

print("Starting training...")
trainer.train()

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✓ Model saved to {OUTPUT_DIR}")

## Cell 4 — Evaluate: Base vs Fine-Tuned

Runs inference with both base model and fine-tuned adapter on 100 test examples.  
Computes: valid-JSON rate, exact match, per-field accuracy.

In [ ]:
import json, re, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"
ADAPTER_DIR = "./lora_model"
FIELDS = ["action", "item", "quantity", "date"]
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_test_data(path="data/test.jsonl"):
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

def make_inference_prompt(inp, tokenizer):
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        msgs = [{"role": "user", "content": inp}]
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return f"Input: {inp}\nOutput:"

def generate(model, tokenizer, prompt, max_new=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

def extract_json(text):
    """Try to parse JSON from model output."""
    text = text.strip()
    # Try direct parse
    try: return json.loads(text)
    except: pass
    # Try extracting first {...}
    m = re.search(r'\{.*?\}', text, re.DOTALL)
    if m:
        try: return json.loads(m.group())
        except: pass
    return None

def evaluate_model(model, tokenizer, test_data):
    model.eval().to(device)
    valid_json = 0
    exact_match = 0
    field_correct = {f: 0 for f in FIELDS}
    field_total = 0
    failures = []

    for ex in test_data:
        prompt = make_inference_prompt(ex["input"], tokenizer)
        output = generate(model, tokenizer, prompt)
        expected = json.loads(ex["output"])

        parsed = extract_json(output)
        if parsed is not None:
            valid_json += 1
            field_total += 1
            # Check exact match (all fields correct)
            if all(str(parsed.get(f,"")).lower() == str(expected.get(f,"")).lower() for f in FIELDS):
                exact_match += 1
            # Per-field
            for f in FIELDS:
                if str(parsed.get(f,"")).lower() == str(expected.get(f,"")).lower():
                    field_correct[f] += 1
                elif len(failures) < 3:
                    reason = f"Field '{f}': expected '{expected.get(f)}', got '{parsed.get(f)}'"
                    failures.append({"input": ex["input"], "expected": ex["output"],
                                      "got": output, "reason": reason})
        else:
            if len(failures) < 3:
                failures.append({"input": ex["input"], "expected": ex["output"],
                                  "got": output, "reason": "Output is not valid JSON"})

    n = len(test_data)
    ft = max(field_total, 1)
    return {
        "valid_json_rate": round(valid_json / n, 4),
        "exact_match": round(exact_match / n, 4),
        "per_field_accuracy": {f: round(field_correct[f] / ft, 4) for f in FIELDS},
    }, failures

test_data = load_test_data()
print(f"Loaded {len(test_data)} test examples.")

# --- Evaluate BASE model ---
print("\nEvaluating BASE model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True
)
base_metrics, base_failures = evaluate_model(base_model, tokenizer, test_data)
print("Base:", base_metrics)
del base_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# --- Evaluate FINE-TUNED model ---
print("\nEvaluating FINE-TUNED model...")
ft_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True
)
ft_model = PeftModel.from_pretrained(ft_base, ADAPTER_DIR)
ft_model = ft_model.merge_and_unload()
ft_metrics, ft_failures = evaluate_model(ft_model, tokenizer, test_data)
print("Fine-tuned:", ft_metrics)

# Pick top 3 failures (prefer fine-tuned failures for analysis)
all_failures = ft_failures[:3] if ft_failures else base_failures[:3]

# Save results
results = {"base": base_metrics, "finetuned": ft_metrics, "failures": all_failures}
with open("results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\n✓ Results saved to results.json")
print(json.dumps(results, indent=2))

## Cell 6b: Evaluate on Hard Test Set (20 adversarial examples)
Typos, missing fields, different phrasing — test robustness.

## Cell 5 — Download results.json

Downloads `results.json` to your local machine.  
Copy it to `task1_lora/results.json` in the repo, then run `python generate_report.py`.

In [ ]:
# ── Evaluate fine-tuned model on hard test set ─────────────────────────────
import json, re, torch

HARD_TEST = "./data/test_hard.jsonl"
hard_examples = [json.loads(l) for l in open(HARD_TEST)]

def run_hard_eval(model, tokenizer, examples, label):
    model.eval()
    valid, exact, field_hits, field_total = 0, 0, {f:0 for f in ["action","item","quantity","date"]}, {f:0 for f in ["action","item","quantity","date"]}
    for ex in examples:
        prompt = f"Convert to JSON: {ex["input"]}\nOutput:"
        ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=80, do_sample=False)
        pred_text = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
        try:
            pred = json.loads(re.search(r"\{.*?\}", pred_text, re.DOTALL).group())
            valid += 1
            gt = ex["output"]
            # normalise
            def norm(v): return str(int(v)).lower() if str(v).isdigit() else str(v).lower()
            if all(norm(pred.get(f)) == norm(gt.get(f,"")) for f in ["action","item","quantity","date"] if gt.get(f) is not None):
                exact += 1
            for f in ["action","item","quantity","date"]:
                if gt.get(f) is not None:
                    field_total[f] = field_total.get(f,0)+1
                    if norm(pred.get(f,"")) == norm(gt.get(f,"")):
                        field_hits[f] = field_hits.get(f,0)+1
        except: pass
    n = len(examples)
    print(f"{label}: valid={valid/n:.1%} exact={exact/n:.1%} per_field={{{', '.join(f+':'+f'{field_hits[f]/max(1,field_total[f]):.0%}' for f in ["action","item","quantity","date"])}}}")
    return {"valid_json_rate": round(valid/n,4), "exact_match": round(exact/n,4)}

hard_base   = run_hard_eval(base_model,       tokenizer, hard_examples, "BASE (hard)")
hard_tuned  = run_hard_eval(finetuned_model,  tokenizer, hard_examples, "TUNED (hard)")

# Append to results.json
with open("results.json") as f: r = json.load(f)
r["hard_test"] = {"base": hard_base, "finetuned": hard_tuned, "n": len(hard_examples)}
with open("results.json","w") as f: json.dump(r, f, indent=2)
print("Hard test results saved to results.json")


In [ ]:
from google.colab import files
import json

# Print a summary first
with open("results.json") as f:
    r = json.load(f)

print("=" * 50)
print("FINAL RESULTS SUMMARY")
print("=" * 50)
print(f"\nBASE MODEL:")
print(f"  Valid JSON rate : {r['base']['valid_json_rate']:.1%}")
print(f"  Exact match     : {r['base']['exact_match']:.1%}")
print(f"  Per-field       : {r['base']['per_field_accuracy']}")
print(f"\nFINE-TUNED MODEL:")
print(f"  Valid JSON rate : {r['finetuned']['valid_json_rate']:.1%}")
print(f"  Exact match     : {r['finetuned']['exact_match']:.1%}")
print(f"  Per-field       : {r['finetuned']['per_field_accuracy']}")
print(f"\nFAILURES: {len(r['failures'])} captured")
print("=" * 50)
print("\nNext steps:")
print("1. Copy results.json to task1_lora/results.json in the repo")
print("2. Run: python generate_report.py")

# Download to local machine
files.download("results.json")
print("\n✓ results.json downloaded!")